---
## 0. Setup

In [2]:
import pandas as pd

print('Merging data...')

Merging data...


In [5]:
users        = pd.read_csv('cleaned_data/users_clean.csv')
movies       = pd.read_csv('cleaned_data/movies_clean.csv')
watch        = pd.read_csv('cleaned_data/watch_history_clean.csv')
reviews      = pd.read_csv('cleaned_data/reviews_clean.csv')
recs         = pd.read_csv('cleaned_data/recommendation_logs_clean.csv')
searches     = pd.read_csv('cleaned_data/search_logs_clean.csv')

In [13]:
# Reference date can be adjusted depending on how exactly we want the model to predict
reference_date = pd.Timestamp('2025-12-31')

---
## 1. Watch Features

In [6]:
watch['watch_date'] = pd.to_datetime(watch['watch_date'])

watch_movies = watch.merge(movies, on='movie_id', how='left')

watch_features = watch_movies.groupby('user_id').agg(
    total_sessions=('session_id', 'count'),
    unique_titles_watched=('movie_id', 'nunique'),
    total_watch_minutes=('watch_duration_minutes', 'sum'),
    avg_watch_duration=('watch_duration_minutes', 'mean'),
    avg_progress=('progress_percentage', 'mean'),
    pct_completed=('action', lambda s: (s == 'completed').mean()),
    download_sessions=('is_download', 'sum'),
    first_watch_date=('watch_date', 'min'),
    last_watch_date=('watch_date', 'max'),
    active_watch_days=('watch_date', lambda s: s.dt.date.nunique()),
    # From movies dataset
    avg_imdb_watched=('imdb_rating', 'mean'),
    unique_genres=('genre_primary', 'nunique'),
    pct_originals=('is_netflix_original', 'mean')
).reset_index()

In [9]:
watch_features['watch_span_days'] = (
    watch_features['last_watch_date'] - watch_features['first_watch_date']
).dt.days + 1
watch_features['sessions_per_active_day'] = (
    watch_features['total_sessions'] / watch_features['active_watch_days']
)
watch_features['sessions_per_month'] = (
    watch_features['total_sessions'] / (watch_features['watch_span_days'] / 30)
)
watch_features['watch_minutes_per_month'] = (
    watch_features['total_watch_minutes'] / (watch_features['watch_span_days'] / 30)
)

In [14]:
watch_features['days_since_last_watch'] = (
    reference_date - watch_features['last_watch_date']
).dt.days

In [15]:
watch_features.head(10)

,user_id,total_sessions,unique_titles_watched,total_watch_minutes,avg_watch_duration,avg_progress,pct_completed,download_sessions,first_watch_date,last_watch_date,active_watch_days,avg_imdb_watched,unique_genres,pct_originals,sessions_per_active_day,watch_span_days,sessions_per_month,watch_minutes_per_month,days_since_last_watch
0,user_00001,12,12,734.7,61.225000,64.516667,0.500000,4,2024-02-05,2025-10-30,12,7.000000,9,0.416667,1.000000,634,0.567823,34.764984,62
1,user_00002,15,15,764.6,50.973333,56.620000,0.333333,4,2024-02-29,2025-11-28,14,5.673333,11,0.533333,1.071429,639,0.704225,35.896714,33
2,user_00003,8,8,594.2,74.275000,43.437500,0.375000,2,2024-04-25,2025-06-24,8,6.987500,6,0.375000,1.000000,426,0.563380,41.845070,190
3,user_00004,14,13,1197.9,85.564286,38.764286,0.285714,3,2024-02-17,2025-12-29,14,5.307143,10,0.357143,1.000000,682,0.615836,52.693548,2
4,user_00005,9,9,563.0,62.555556,54.444444,0.222222,2,2024-01-25,2025-09-19,9,6.166667,9,0.222222,1.000000,604,0.447020,27.963576,103
5,user_00006,11,11,479.2,43.563636,62.018182,0.272727,4,2024-02-25,2025-11-30,11,7.004545,10,0.363636,1.000000,645,0.511628,22.288372,31
6,user_00007,12,12,713.4,59.450000,39.791667,0.166667,3,2024-08-07,2025-12-30,12,6.429167,10,0.333333,1.000000,511,0.704501,41.882583,1
7,user_00008,11,11,512.3,46.572727,38.790909,0.181818,1,2024-02-05,2025-11-27,11,6.090909,9,0.363636,1.000000,662,0.498489,23.216012,34
8,user_00009,7,7,439.2,62.742857,47.614286,0.000000,0,2024-03-22,2025-11-23,7,6.171429,6,0.285714,1.000000,612,0.343137,21.529412,38
9,user_00010,8,8,498.7,62.337500,55.700000,0.125000,2,2024-03-09,2025-11-18,8,5.956250,8,0.250000,1.000000,620,0.387097,24.130645,43


---
## 2. Reviews Features

In [16]:
reviews['review_date'] = pd.to_datetime(reviews['review_date'])

review_features = reviews.groupby('user_id').agg(
    review_count=('review_id', 'count'),
    avg_review_rating=('rating', 'mean'),
    avg_sentiment=('sentiment_score', 'mean'),
    pct_positive_reviews=('sentiment', lambda s: (s == 'positive').mean()),
    last_review_date=('review_date', 'max'),
).reset_index()

In [17]:
last_positive = (
    reviews[reviews['sentiment'] == 'positive']
    .groupby('user_id')['review_date']
    .max()
    .reset_index(name='last_positive_review_date')
)

review_features = review_features.merge(last_positive, on='user_id', how='left')

review_features['days_since_last_review'] = (
    reference_date - review_features['last_review_date']
).dt.days

review_features['days_since_last_positive_review'] = (
    reference_date - review_features['last_positive_review_date']
).dt.days


In [18]:
review_features.head(10)

,user_id,review_count,avg_review_rating,avg_sentiment,pct_positive_reviews,last_review_date,last_positive_review_date,days_since_last_review,days_since_last_positive_review
0,user_00001,1,4.000000,0.792000,1.000000,2024-01-09,2024-01-09,722,722.0
1,user_00002,4,3.500000,0.537500,0.500000,2025-09-12,2025-07-14,110,170.0
2,user_00003,1,4.000000,0.887000,1.000000,2024-04-17,2024-04-17,623,623.0
3,user_00004,3,2.666667,0.476067,0.333333,2025-03-12,2024-08-22,294,496.0
4,user_00005,3,3.333333,0.502667,0.333333,2025-09-26,2024-06-20,96,559.0
5,user_00006,1,3.000000,0.506000,0.000000,2025-01-30,NaT,335,NaN
6,user_00007,3,4.000000,0.702667,0.666667,2024-10-17,2024-10-17,440,440.0
7,user_00010,2,2.500000,0.501500,0.500000,2025-03-25,2025-03-25,281,281.0
8,user_00011,1,5.000000,0.829000,1.000000,2024-07-14,2024-07-14,535,535.0
9,user_00013,2,3.500000,0.601000,0.500000,2025-11-29,2025-11-29,32,32.0


---
## 3. Recommendation Features

In [19]:
recs['recommendation_date'] = pd.to_datetime(recs['recommendation_date'])

rec_features = recs.groupby('user_id').agg(
    rec_count=('recommendation_id', 'count'),
    rec_click_rate=('was_clicked', 'mean'),
    avg_rec_score=('recommendation_score', 'mean'),
    last_rec_date=('recommendation_date', 'max'),
).reset_index()

In [20]:
rec_features['days_since_last_rec'] = (
    reference_date - rec_features['last_rec_date']
).dt.days

last_rec_click = (
    recs[recs['was_clicked'] == True]
    .groupby('user_id')['recommendation_date']
    .max()
    .reset_index(name='last_rec_click_date')
)

rec_features = rec_features.merge(last_rec_click, on='user_id', how='left')

rec_features['days_since_last_rec_click'] = (
    reference_date - rec_features['last_rec_click_date']
).dt.days

In [21]:
rec_features.head(10)

,user_id,rec_count,rec_click_rate,avg_rec_score,last_rec_date,days_since_last_rec,last_rec_click_date,days_since_last_rec_click
0,user_00001,3,0.000000,0.315000,2025-04-26,249,NaT,NaN
1,user_00003,6,0.333333,0.640833,2025-04-13,262,2025-04-13,262.0
2,user_00004,4,0.000000,0.589000,2025-08-10,143,NaT,NaN
3,user_00005,1,0.000000,0.473000,2024-04-02,638,NaT,NaN
4,user_00006,7,0.000000,0.562143,2025-12-12,19,NaT,NaN
5,user_00007,2,0.000000,0.609500,2024-12-17,379,NaT,NaN
6,user_00008,3,0.000000,0.741000,2024-11-24,402,NaT,NaN
7,user_00009,5,0.000000,0.439600,2024-10-23,434,NaT,NaN
8,user_00010,5,0.000000,0.611200,2025-05-10,235,NaT,NaN
9,user_00011,5,0.400000,0.538000,2025-01-20,345,2025-01-20,345.0


---
## 4. Search Features


In [22]:
searches['search_date'] = pd.to_datetime(searches['search_date'])

search_features = searches.groupby('user_id').agg(
    search_count=('search_id', 'count'),
    search_click_rate=('clicked_result_position', lambda s: (s > 0).mean()),
    avg_click_position=('clicked_result_position', lambda s: s[s > 0].mean()),
    avg_search_duration=('search_duration_seconds', 'mean'),
    first_search_date=('search_date', 'min'),
    last_search_date=('search_date', 'max'),
    active_search_days=('search_date', lambda s: s.dt.date.nunique()),
).reset_index()

In [23]:
search_features['search_span_days'] = (
    search_features['last_search_date'] - search_features['first_search_date']
).dt.days + 1

search_features['days_since_last_search'] = (
    reference_date - search_features['last_search_date']
).dt.days

search_features['searches_per_month'] = (
    search_features['search_count'] / (search_features['search_span_days'] / 30)
)

search_features['searches_per_active_day'] = (
    search_features['search_count'] / search_features['active_search_days']
)

last_search_click = (
    searches[searches['clicked_result_position'] > 0]
    .groupby('user_id')['search_date']
    .max()
    .reset_index(name='last_search_click_date')
)

search_features = search_features.merge(last_search_click, on='user_id', how='left')

search_features['days_since_last_search_click'] = (
    reference_date - search_features['last_search_click_date']
).dt.days

In [24]:
search_features.head(10)

,user_id,search_count,search_click_rate,avg_click_position,avg_search_duration,first_search_date,last_search_date,active_search_days,search_span_days,days_since_last_search,searches_per_month,searches_per_active_day,last_search_click_date,days_since_last_search_click
0,user_00001,1,1.0,7.00,27.50,2024-12-29,2024-12-29,1,1,367,30.000000,1.0,2024-12-29,367.0
1,user_00002,3,0.0,NaN,18.10,2024-01-28,2025-12-08,3,681,23,0.132159,1.0,NaT,NaN
2,user_00004,2,0.0,NaN,26.55,2024-11-20,2025-12-29,2,405,2,0.148148,1.0,NaT,NaN
3,user_00005,1,1.0,10.00,37.70,2025-09-29,2025-09-29,1,1,93,30.000000,1.0,2025-09-29,93.0
4,user_00006,1,0.0,NaN,17.40,2024-06-11,2024-06-11,1,1,568,30.000000,1.0,NaT,NaN
5,user_00007,2,0.0,NaN,10.65,2024-04-17,2024-08-06,2,112,512,0.535714,1.0,NaT,NaN
6,user_00008,2,0.5,3.00,15.40,2024-04-04,2024-08-04,2,123,514,0.487805,1.0,2024-08-04,514.0
7,user_00009,2,0.5,3.00,39.10,2024-10-14,2024-12-08,2,56,388,1.071429,1.0,2024-10-14,443.0
8,user_00010,3,0.0,NaN,15.70,2024-06-04,2025-12-04,3,549,27,0.163934,1.0,NaT,NaN
9,user_00012,5,0.8,6.75,21.12,2024-04-26,2025-12-23,5,607,8,0.247117,1.0,2025-12-23,8.0


## 5. Merge

In [25]:
users['created_at'] = pd.to_datetime(users['created_at'])
users['subscription_start_date'] = pd.to_datetime(users['subscription_start_date'])

In [26]:
users['tenure_days'] = (reference_date - users['created_at']).dt.days + 1


In [27]:
df = users.merge(watch_features, on='user_id', how='left')
df = df.merge(review_features, on='user_id', how='left')
df = df.merge(rec_features, on='user_id', how='left')
df = df.merge(search_features, on='user_id', how='left')

In [29]:
df.describe()

,age,subscription_start_date,monthly_spend,household_size,created_at,tenure_days,total_sessions,unique_titles_watched,total_watch_minutes,avg_watch_duration,...,avg_search_duration,first_search_date,last_search_date,active_search_days,search_span_days,days_since_last_search,searches_per_month,searches_per_active_day,last_search_click_date,days_since_last_search_click
count,10000.000000,10000,10000.000000,10000.000000,10000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,...,9186.000000,9186,9186,9186.000000,9186.000000,9186.000000,9186.000000,9186.000000,6851,6851.000000
mean,35.945083,2024-01-31 17:37:40.800000,21.526802,2.865298,2024-02-04 03:02:25.221875,696.371800,10.000000,9.947000,639.941760,64.088897,...,19.942146,2024-08-16 08:09:24.337034,2025-05-15 23:42:45.382103,2.716634,273.648160,229.011975,7.301577,1.001534,2025-03-09 07:33:47.791563,296.684864
min,13.000000,2022-08-02 00:00:00,0.110000,1.000000,2022-08-03 08:42:45.250770,151.000000,2.000000,2.000000,5.500000,2.750000,...,0.300000,2024-01-01 00:00:00,2024-01-01 00:00:00,1.000000,1.000000,0.000000,0.085837,1.000000,2024-01-01 00:00:00,0.000000
25%,29.000000,2023-05-02 00:00:00,8.375000,2.000000,2023-05-07 02:22:31.400241,423.000000,8.000000,8.000000,439.475000,50.642560,...,13.450000,2024-03-17 00:00:00,2025-01-24 00:00:00,2.000000,28.000000,78.000000,0.223214,1.000000,2024-10-06 00:00:00,124.000000
50%,35.767811,2024-01-31 00:00:00,14.469887,2.857143,2024-02-07 19:36:16.319902,693.000000,10.000000,10.000000,594.300000,59.675000,...,18.400000,2024-07-01 00:00:00,2025-07-02 00:00:00,2.000000,273.000000,182.000000,0.350877,1.000000,2025-04-09 00:00:00,266.000000
75%,42.000000,2024-11-04 00:00:00,20.420000,4.000000,2024-11-03 19:42:30.355795,969.000000,12.000000,12.000000,790.325000,71.227083,...,24.600000,2024-12-14 00:00:00,2025-10-14 00:00:00,4.000000,459.750000,341.000000,2.307692,1.000000,2025-08-29 00:00:00,451.000000
max,109.000000,2025-08-01 00:00:00,997.800000,8.000000,2025-08-02 14:44:19.470061,1246.000000,23.000000,22.000000,2535.900000,252.725000,...,103.100000,2025-12-31 00:00:00,2025-12-31 00:00:00,9.000000,731.000000,730.000000,60.000000,2.000000,2025-12-31 00:00:00,730.000000
std,10.803069,NaN,62.805681,1.442294,NaN,316.210992,3.161044,3.134928,284.172814,21.646090,...,9.748149,NaN,NaN,1.460770,219.268039,183.514308,12.470239,0.024595,NaN,202.921882


In [30]:
df.head(10)

,user_id,email,first_name,last_name,age,gender,country,state_province,city,subscription_plan,...,avg_search_duration,first_search_date,last_search_date,active_search_days,search_span_days,days_since_last_search,searches_per_month,searches_per_active_day,last_search_click_date,days_since_last_search_click
0,user_00001,figueroajohn@example.org,Erica,Garza,43.000000,Male,USA,Massachusetts,North Jefferyhaven,Basic,...,27.50,2024-12-29,2024-12-29,1.0,1.0,367.0,30.000000,1.0,2024-12-29,367.0
1,user_00002,blakeerik@example.com,Joshua,Bernard,38.000000,Male,USA,Texas,North Noahstad,Premium+,...,18.10,2024-01-28,2025-12-08,3.0,681.0,23.0,0.132159,1.0,NaT,NaN
2,user_00003,smiller@example.net,Barbara,Williams,32.000000,Female,USA,Michigan,Traciebury,Standard,...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN
3,user_00004,mitchellclark@example.com,Chelsea,Ferguson,36.269641,Male,USA,Ohio,South Noah,Standard,...,26.55,2024-11-20,2025-12-29,2.0,405.0,2.0,0.148148,1.0,NaT,NaN
4,user_00005,richard13@example.net,Jason,Foster,21.000000,Female,USA,Arizona,West Donald,Standard,...,37.70,2025-09-29,2025-09-29,1.0,1.0,93.0,30.000000,1.0,2025-09-29,93.0
5,user_00006,jeffrey28@example.com,Katelyn,Hernandez,43.000000,Female,Canada,Quebec,Carlsonfurt,Standard,...,17.40,2024-06-11,2024-06-11,1.0,1.0,568.0,30.000000,1.0,NaT,NaN
6,user_00007,maldonadoamanda@example.com,Daniel,Moore,63.000000,Unknown,USA,Maryland,Teresaburgh,Premium,...,10.65,2024-04-17,2024-08-06,2.0,112.0,512.0,0.535714,1.0,NaT,NaN
7,user_00008,nathanielmartin@example.net,Carol,Smith,35.712952,Male,Canada,New Brunswick,Johnbury,Basic,...,15.40,2024-04-04,2024-08-04,2.0,123.0,514.0,0.487805,1.0,2024-08-04,514.0
8,user_00009,donnaarroyo@example.org,Allison,Brandt,24.000000,Male,USA,North Carolina,Raymondstad,Premium,...,39.10,2024-10-14,2024-12-08,2.0,56.0,388.0,1.071429,1.0,2024-10-14,443.0
9,user_00010,morganbrianna@example.org,Adrian,Perez,24.000000,Male,USA,Pennsylvania,Chadbury,Basic,...,15.70,2024-06-04,2025-12-04,3.0,549.0,27.0,0.163934,1.0,NaT,NaN


In [33]:
# Fill count-based features with 0 for users with no activity
count_cols = [
    'total_sessions', 'review_count', 'rec_count', 'search_count',
    'download_sessions', 'active_watch_days', 'active_search_days',
]
df[count_cols] = df[count_cols].fillna(0)

In [34]:
# Activity flags (before recency fills)
df['has_watch_history'] = df['total_sessions'].gt(0).astype(int)
df['has_reviews'] = df['review_count'].gt(0).astype(int)
df['has_recs'] = df['rec_count'].gt(0).astype(int)
df['has_searches'] = df['search_count'].gt(0).astype(int)

In [35]:
# Recency: never engaged = very long time since
recency_cols = [c for c in df.columns if c.startswith('days_since_')]
df[recency_cols] = df[recency_cols].fillna(9999)

In [36]:
# Rates / averages / frequency
fill_zero_cols = [
    'pct_completed', 'pct_originals', 'pct_positive_reviews',
    'rec_click_rate', 'search_click_rate',
    'avg_review_rating', 'avg_sentiment', 'avg_rec_score',
    'avg_click_position', 'avg_search_duration',
    'avg_watch_duration', 'avg_progress', 'avg_imdb_watched',
    'sessions_per_active_day', 'sessions_per_month', 'watch_minutes_per_month',
    'searches_per_month', 'searches_per_active_day',
    'watch_span_days',
]
df[fill_zero_cols] = df[fill_zero_cols].fillna(0)

In [37]:
# Verify
print(df.isnull().sum().sort_values(ascending=False).head(20))

last_rec_click_date                4701
last_positive_review_date          4070
last_search_click_date             3149
last_review_date                   2239
last_search_date                    814
first_search_date                   814
search_span_days                    814
last_rec_date                        79
user_id                               0
avg_rec_score                         0
rec_click_rate                        0
days_since_last_positive_review       0
rec_count                             0
days_since_last_review                0
pct_positive_reviews                  0
avg_sentiment                         0
avg_review_rating                     0
days_since_last_rec                   0
search_click_rate                     0
days_since_last_rec_click             0
dtype: int64


In [39]:
df.to_csv('cleaned_data/users_merged.csv', index=False)